# 📝 Зачётная работа: классификатор новостей

**Датасет ваш:** новости трёх направлений из домашнего задания. **Его обязательно приложить к работе.**

Нужен файл `news.csv` с двумя столбцами: `text` и `label`. Перетащите его в панель «Файлы» слева.

Сдать: **Файл → Скачать .ipynb** + `news.csv` (или ссылка на Colab с доступом по ссылке).

| Часть | Баллы LXP |
|---|---|
| Две сети, лучшая модель, точность на test | 5 |
| Нормализация слов (задание 3) | ещё 5 |

Ноутбук проверяет форму результата. Точность на вашем датасете зависит от данных, это нормально.

In [ ]:
#@title ⚙️ Запустите первой: библиотеки и проверки { display-mode: "form" }
!pip install -q pymorphy3 nltk
import os, re
from collections import Counter
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
torch.manual_seed(42); np.random.seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

def check(tests, n):
    for i, (name, cond, hint) in enumerate(tests, 1):
        try:
            passed = bool(cond())
        except Exception:
            passed = False
        if not passed:
            print(f"❌ Тест {i} из {len(tests)}: {name}\n   💡 {hint}")
            return False
        print(f"✅ Тест {i}: {name}")
    print(f"\n🎉 Задание {n} выполнено!")
    return True

print("✅ Готово")

---
# 1. Загрузка и EDA

### ✏️ Задание 1. Загрузите `news.csv` в `df`, удалите пропуски и дубликаты текстов

In [ ]:
df = None
df = None

In [ ]:
#@title 🧪 Проверка задания 1 { display-mode: "form" }
if check([
    ('столбцы', lambda: set(["text", "label"]) <= set(df.columns), 'В файле нужны столбцы text и label'),
    ('три класса', lambda: df["label"].nunique() == 3, 'По заданию нужны три направления'),
    ('чисто', lambda: df["text"].isna().sum() == 0 and df["text"].duplicated().sum() == 0, 'dropna и drop_duplicates'),
], 1):
    pass

### ✏️ Задание 2. EDA: `class_counts` (текстов в классе) и `text_len` (длина каждого текста в словах). Постройте 2 графика

In [ ]:
class_counts = None
text_len = None
pass  # постройте столбчатую диаграмму классов и гистограмму длин

In [ ]:
#@title 🧪 Проверка задания 2 { display-mode: "form" }
if check([
    ('классы', lambda: class_counts.sum() == len(df), 'value_counts по label'),
    ('длины', lambda: len(text_len) == len(df), 'Длина для каждого текста'),
], 2):
    print(class_counts); print("Средняя длина:", round(text_len.mean()))

---
# 2. Токенизация и нормализация

Запишите ваш вариант в `VARIANT`: `"lemma"` (лемматизация) или `"stem"` (стемминг).

```python
# Лемматизация: начальная форма
import pymorphy3
morph = pymorphy3.MorphAnalyzer()
morph.parse("новостями")[0].normal_form   # 'новость'

# Стемминг: грубо отрезает окончание
from nltk.stem.snowball import SnowballStemmer
stemmer = SnowballStemmer("russian")
stemmer.stem("новостями")                 # 'новост'
```

### ✏️ Задание 3. Напишите `tokenize(text)` (нижний регистр, только буквы) и `normalize(word)` по своему варианту

In [ ]:
VARIANT = None

def tokenize(text):
    return None

pass  # создайте morph или stemmer

cache = {}
def normalize(word):
    if word not in cache:
        cache[word] = None
    return cache[word]

df["tokens"] = df["text"].apply(lambda t: [normalize(w) for w in tokenize(t)])

In [ ]:
#@title 🧪 Проверка задания 3 { display-mode: "form" }
if check([
    ('вариант', lambda: VARIANT in ("lemma", "stem"), 'Запишите "lemma" или "stem"'),
    ('токены', lambda: tokenize("Кот, спит! 123") == ["кот", "спит"], 'Нижний регистр и только буквы'),
    ('нормализация', lambda: normalize("новостями") in ("новость", "новост"), 'Используйте пример выше'),
], 3):
    print(df["tokens"].iloc[0][:15])
    print("Разных слов:", len({w for t in df["tokens"] for w in t}))

---
# 3. Выборки

### ✏️ Задание 4. Разделите на `xTrain`, `xVal`, `xTest` (70/15/15) и получите `yTrain`, `yVal`, `yTest`

In [ ]:
classes = sorted(df["label"].unique())
y = df["label"].map({c: i for i, c in enumerate(classes)}).values
tokens = df["tokens"].tolist()
xTrain, xRest, yTrain, yRest = None, None, None, None
xVal, xTest, yVal, yTest = None, None, None, None

In [ ]:
#@title 🧪 Проверка задания 4 { display-mode: "form" }
if check([
    ('доля train', lambda: abs(len(xTrain) / len(tokens) - 0.7) < 0.02, 'test_size=0.3'),
    ('val и test', lambda: abs(len(xVal) - len(xTest)) <= 1, 'Остаток пополам'),
    ('пары совпадают', lambda: len(xTrain) == len(yTrain) and len(xTest) == len(yTest), 'Делите x и y вместе'),
], 4):
    pass

### ✏️ Задание 5. Постройте словарь `word2idx` по **xTrain**: 5000 самых частых слов, номера с 2 (0 — PAD, 1 — UNK)

In [ ]:
MAX_WORDS = 5000
counts = Counter(w for t in xTrain for w in t)
word2idx = None
VOCAB = len(word2idx) + 2

In [ ]:
#@title 🧪 Проверка задания 5 { display-mode: "form" }
if check([
    ('номера', lambda: min(word2idx.values()) == 2, 'Нумерация с 2'),
    ('размер', lambda: len(word2idx) <= MAX_WORDS, 'most_common(MAX_WORDS)'),
], 5):
    pass

---
# 4. Сеть на мешке слов

### ✏️ Задание 6. Напишите `to_bow(texts)`: матрица float32 размера (текстов, VOCAB) со счётчиками слов

In [ ]:
def to_bow(texts):
    M = np.zeros((len(texts), VOCAB), dtype="float32")
    for i, t in enumerate(texts):
        for w in t:
            pass
    return torch.tensor(M)

bTrain, bVal, bTest = to_bow(xTrain), to_bow(xVal), to_bow(xTest)

In [ ]:
#@title 🧪 Проверка задания 6 { display-mode: "form" }
if check([
    ('форма', lambda: bTrain.shape == (len(xTrain), VOCAB), 'Строка на текст, столбец на слово'),
    ('счётчики', lambda: bTrain.sum() > 0, 'Прибавляйте 1 в нужную ячейку'),
], 6):
    pass

Функция обучения общая для двух сетей. **В ней не хватает сохранения лучшей модели:** это аналог колбэка ModelCheckpoint.

### ✏️ Задание 7. Допишите в `fit` сохранение лучших весов и обучите `bow_net` (не меньше двух слоёв `Linear`)

In [ ]:
def fit(net, X, Y, Xv, Yv, path, epochs=15, lr=0.001):
    net = net.to(device)
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    Y, Yv = torch.tensor(Y), torch.tensor(Yv)
    best = -1
    for ep in range(1, epochs + 1):
        net.train()
        for xb, yb in DataLoader(TensorDataset(X, Y), batch_size=32, shuffle=True):
            loss = loss_fn(net(xb.to(device)), yb.to(device))
            opt.zero_grad(); loss.backward(); opt.step()
        net.eval()
        with torch.no_grad():
            acc = (net(Xv.to(device)).argmax(1).cpu() == Yv).float().mean().item()
        print(f"эпоха {ep}: val accuracy {acc:.3f}")
        pass  # если acc лучше best: обновите best и сохраните веса в path
    return best

bow_net = None
best_bow = fit(bow_net, bTrain, yTrain, bVal, yVal, "best_bow.pt")

In [ ]:
#@title 🧪 Проверка задания 7 { display-mode: "form" }
if check([
    ('файл весов', lambda: os.path.exists("best_bow.pt"), 'torch.save(net.state_dict(), path)'),
    ('выход на 3 класса', lambda: bow_net(bVal[:2].to(device)).shape == (2, 3), 'Последний слой на 3 выхода'),
    ('лучше случайного', lambda: best_bow > 0.4, 'Наугад было бы 0.33'),
], 7):
    pass

---
# 5. Сеть с Embedding

### ✏️ Задание 8. Напишите `to_seq(texts)`: тензор номеров длины 200, и обучите `emb_net`

In [ ]:
SEQ = 200
def to_seq(texts):
    rows = []
    for t in texts:
        ids = [word2idx.get(w, 1) for w in t][:SEQ]
        rows.append(None)  # дополните нулями до SEQ
    return torch.tensor(rows)

sTrain, sVal, sTest = to_seq(xTrain), to_seq(xVal), to_seq(xTest)

class EmbNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = None
        self.head = None
    def forward(self, x):
        e = self.emb(x)
        mean = e.sum(1) / (x != 0).sum(1, keepdim=True).clamp(min=1)   # средний вектор слов текста
        return self.head(mean)

emb_net = EmbNet()
best_emb = fit(emb_net, sTrain, yTrain, sVal, yVal, "best_emb.pt", epochs=25, lr=0.003)

In [ ]:
#@title 🧪 Проверка задания 8 { display-mode: "form" }
if check([
    ('длина', lambda: sTrain.shape[1] == SEQ, 'Все строки длины SEQ'),
    ('файл весов', lambda: os.path.exists("best_emb.pt"), 'Запустите fit'),
    ('лучше случайного', lambda: best_emb > 0.4, 'Наугад было бы 0.33'),
], 8):
    pass

---
# 6. Итог

### ✏️ Задание 9. Загрузите лучшие веса обеих сетей и посчитайте точность на test: `acc_bow`, `acc_emb`

In [ ]:
def test_acc(net, X, path):
    return None  # загрузите веса из path, переведите сеть в eval, посчитайте точность на yTest

acc_bow = test_acc(bow_net, bTest, "best_bow.pt")
acc_emb = test_acc(emb_net, sTest, "best_emb.pt")

In [ ]:
#@title 🧪 Проверка задания 9 { display-mode: "form" }
if check([
    ('BoW', lambda: 0.4 < acc_bow <= 1, 'load_state_dict(torch.load(path))'),
    ('Embedding', lambda: 0.4 < acc_emb <= 1, 'Та же функция'),
], 9):
    print(f"Мешок слов: {acc_bow:.3f}")
    print(f"Embedding: {acc_emb:.3f}")

**Вывод (2–3 предложения):** какая сеть оказалась лучше и почему? Как повлияла нормализация слов?

_Ваш ответ:_